# Bases de connaissance : interroger Wikidata à l'aide de requêtes SPARQL

## Imports

In [6]:
import time
from datetime import datetime as dt
from urllib.error import HTTPError
from SPARQLWrapper import SPARQLWrapper, JSON

## Obtenir la liste des politiciens belges

In [ ]:
# Retrieve results from SPARQL
endpoint = "https://query.wikidata.org/sparql"

sparql = SPARQLWrapper(
    endpoint,
    agent="ULB-TAC-course (educational use); contact: denis.lebailly@ulb.be)",
)

# P27 = country of citizenship 
# Q31 = Belgium
# P106 = Opccupation
# Q82955 = Politician
# P569 = date of birth
# P570 = date of death
statement = """
SELECT DISTINCT ?person ?personLabel ?dateBirth ?dateDeath WHERE {
    ?person wdt:P27 wd:Q31 .
    ?person wdt:P106 wd:Q82955 .
    ?person wdt:P569 ?dateBirth .
    OPTIONAL {?person wdt:P570 ?dateDeath .}
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}
ORDER BY ?personLabel
"""

sparql.setQuery(statement)
sparql.setReturnFormat(JSON)
sparql.setTimeout(120)

# The public Wikidata endpoint occasionally returns transient 429/502/504 errors
# under load, so retry a couple of times before giving up.
max_attempts = 3
for attempt in range(1, max_attempts + 1):
    try:
        results = sparql.query().convert()
        break
    except HTTPError as e:
        if attempt == max_attempts:
            raise
        print(f"Wikidata request failed ({e}), retrying ({attempt}/{max_attempts})...")
        time.sleep(30)

rows = results['results']['bindings']
print(f"\n{len(rows)} Belgian politicians found\n")
print(rows[:10])


8211 Belgian politicians found

[{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q100324607'}, 'personLabel': {'type': 'literal', 'value': 'Q100324607'}, 'dateBirth': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1830-04-01T00:00:00Z'}, 'dateDeath': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1913-01-01T00:00:00Z'}}, {'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q12724173'}, 'personLabel': {'type': 'literal', 'value': 'Q12724173'}, 'dateBirth': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1931-11-11T00:00:00Z'}, 'dateDeath': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '2019-04-24T00:00:00Z'}}, {'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q128636012'}, 'personLabel': {'type': 'literal', 'value': 'Q128636012'}, 'dateBirth': {'datatype': 'http://www.w3.org/20

## Filtrer pour n'afficher que les noms contenant "Alphonse"

In [8]:
name_filter = 'Alphonse'
max_results = 10

date_format = "%Y-%m-%dT%H:%M:%SZ"
filtered_rows = [row for row in rows if name_filter in row['personLabel']['value']]
print(f"Displaying the first {max_results}:\n")
for row in filtered_rows[:max_results]:
    try:
        birth_date = dt.strptime(row['dateBirth']['value'], date_format)
        birth_year = birth_date.year
    except ValueError:
        birth_year = "????"
    try:
        death_date = dt.strptime(row['dateDeath']['value'], date_format)
        death_year = death_date.year
    except ValueError: # unknown death date
        death_year = "????"
    except KeyError: # still alive
        death_year = ""
    print(f"{row['personLabel']['value']} ({birth_year}-{death_year})")

Displaying the first 10:

Adolphe-Alphonse Goupy de Beauvolers (1796-1869)
Alphonse Bonenfant (1908-1983)
Alphonse Bosch (1886-1944)
Alphonse Brenez (1862-1933)
Alphonse Briart (1864-1936)
Alphonse Carpentier (1869-1934)
Alphonse Casse (1846-1923)
Alphonse Clignez (1886-1965)
Alphonse Collet (1905-1971)
Alphonse De Haene (1863-1939)


## Pour en savoir plus

- [Le projet Wikidata](https://www.wikidata.org/wiki/Wikidata:Main_Page)
- [Aide à la construction de requêtes](https://query.wikidata.org/)